# 03 - INSERT, UPDATE e DELETE em Delta no MinIO

Demonstramos as três operações em produtos, uma das cinco tabelas
Delta do bucket bronze. As consultas mostram os resultados antes
e depois de cada alteração.

As alterações são realizadas nas tabelas Delta. Os CSVs do
landing-zone e as tabelas do SQL Server permanecem como origem.

## 1. Iniciar o Spark

Utilizamos as mesmas configurações de Delta Lake e MinIO do
notebook de conversão.

In [1]:
import os
from pathlib import Path
import boto3
from botocore.config import Config
from botocore.exceptions import ClientError
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from delta.tables import DeltaTable

raiz = Path.cwd()
if not (raiz / "pyproject.toml").is_file():
    raiz = raiz.parent

load_dotenv(raiz / ".env", override=True)

s3 = boto3.client(
    "s3",
    endpoint_url=os.environ["MINIO_ENDPOINT"],
    aws_access_key_id=os.environ["MINIO_ROOT_USER"],
    aws_secret_access_key=os.environ["MINIO_ROOT_PASSWORD"],
    region_name="us-east-1",
    config=Config(
        signature_version="s3v4",
        s3={"addressing_style": "path"}
    )
)

spark = (
    SparkSession.builder
    .appName("Loja - DML em Delta no MinIO")
    .master("local[2]")
    .config(
        "spark.jars.packages",
        "io.delta:delta-spark_2.12:3.2.0,"
        "org.apache.hadoop:hadoop-aws:3.3.4"
    )
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
    .config("spark.hadoop.fs.s3a.endpoint", os.environ["MINIO_ENDPOINT"])
    .config("spark.hadoop.fs.s3a.access.key", os.environ["MINIO_ROOT_USER"])
    .config("spark.hadoop.fs.s3a.secret.key", os.environ["MINIO_ROOT_PASSWORD"])
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    )
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.databricks.delta.snapshotPartitions", "2")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Versão do Spark:", spark.version)

26/10/13 19:34:52 WARN Utils: Your hostname, YasminNitroV15 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/13 19:34:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /home/bfontanayasmin/.ivy2/cache
The jars for the packages stored in: /home/bfontanayasmin/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-29e4541c-dc8a-44ca-9e4c-3d721fa1223a;1.0
	confs: [default]


:: loading settings :: url = jar:file:/home/bfontanayasmin/projetos/trabalhoBancodeDados2/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 183ms :: artifacts dl 6ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	--

26/10/13 19:34:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Versão do Spark: 3.5.3


## 2. Preparar a demonstração e registrar as tabelas

Restauramos produtos a partir do CSV para iniciar cada demonstração
com os dez registros originais.

Criamos o namespace loja_bronze e registramos as cinco tabelas
com LOCATION apontando para seus arquivos no MinIO.

Essas tabelas são externas: seus arquivos já existem e o catálogo
registra onde encontrá-los.

In [2]:
caminho_produtos = "s3a://bronze/produtos"

(
    spark.read
    .schema(
        "id_produto INT, nome_produto STRING, "
        "preco DECIMAL(10,2), estoque INT, id_categoria INT"
    )
    .option("header", "true")
    .option("mode", "FAILFAST")
    .csv("s3a://landing-zone/produtos.csv")
    .write
    .format("delta")
    .mode("overwrite")
    .save(caminho_produtos)
)

spark.sql("""
    CREATE DATABASE IF NOT EXISTS loja_bronze
    LOCATION 's3a://bronze/'
""")

tabelas = ["categorias", "clientes", "itens_pedido", "pedidos", "produtos"]

for tabela in tabelas:
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS loja_bronze.{tabela}
        USING DELTA
        LOCATION 's3a://bronze/{tabela}'
    """)

spark.sql("SHOW TABLES IN loja_bronze").show(truncate=False)

assert spark.table("loja_bronze.produtos").count() == 10
print("Demonstração preparada com dez produtos.")

26/10/13 19:34:55 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


26/10/13 19:34:59 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----------+------------+-----------+
|namespace  |tableName   |isTemporary|
+-----------+------------+-----------+
|loja_bronze|categorias  |false      |
|loja_bronze|clientes    |false      |
|loja_bronze|itens_pedido|false      |
|loja_bronze|pedidos     |false      |
|loja_bronze|produtos    |false      |
+-----------+------------+-----------+



Demonstração preparada com dez produtos.


## 3. Inserção — INSERT

Inserimos o Monitor, com identificador 11, preço de R$ 449,90,
estoque de sete unidades e categoria Informática.

A quantidade de produtos passa de dez para onze.

In [3]:
print("Antes do INSERT:")
spark.sql("""
    SELECT * FROM loja_bronze.produtos ORDER BY id_produto
""").show(truncate=False)

spark.sql("""
    INSERT INTO loja_bronze.produtos
        (id_produto, nome_produto, preco, estoque, id_categoria)
    VALUES (11, 'Monitor', 449.90, 7, 1)
""")

print("Depois do INSERT:")
spark.sql("""
    SELECT * FROM loja_bronze.produtos ORDER BY id_produto
""").show(truncate=False)

assert spark.table("loja_bronze.produtos").count() == 11

Antes do INSERT:


+----------+------------+------+-------+------------+
|id_produto|nome_produto|preco |estoque|id_categoria|
+----------+------------+------+-------+------------+
|1         |Mouse       |49.90 |20     |1           |
|2         |Teclado     |120.00|10     |1           |
|3         |Caderno     |19.90 |30     |2           |
|4         |Caneta      |3.50  |100    |2           |
|5         |Headset     |199.90|12     |1           |
|6         |Webcam      |149.90|8      |1           |
|7         |Pendrive    |39.90 |25     |1           |
|8         |Lapiseira   |9.90  |40     |2           |
|9         |Borracha    |2.50  |60     |2           |
|10        |Marcador    |6.90  |35     |2           |
+----------+------------+------+-------+------------+



Depois do INSERT:


+----------+------------+------+-------+------------+
|id_produto|nome_produto|preco |estoque|id_categoria|
+----------+------------+------+-------+------------+
|1         |Mouse       |49.90 |20     |1           |
|2         |Teclado     |120.00|10     |1           |
|3         |Caderno     |19.90 |30     |2           |
|4         |Caneta      |3.50  |100    |2           |
|5         |Headset     |199.90|12     |1           |
|6         |Webcam      |149.90|8      |1           |
|7         |Pendrive    |39.90 |25     |1           |
|8         |Lapiseira   |9.90  |40     |2           |
|9         |Borracha    |2.50  |60     |2           |
|10        |Marcador    |6.90  |35     |2           |
|11        |Monitor     |449.90|7      |1           |
+----------+------------+------+-------+------------+



## 4. Atualização — UPDATE

Repetimos a atualização do trabalho anterior: o Mouse passa de
R$ 49,90 para R$ 59,90, e seu estoque passa de 20 para 15 unidades.

WHERE limita a atualização ao produto com identificador 1.
A quantidade de produtos permanece onze.

In [4]:
print("Antes do UPDATE:")
spark.sql("""
    SELECT * FROM loja_bronze.produtos WHERE id_produto = 1
""").show(truncate=False)

spark.sql("""
    UPDATE loja_bronze.produtos
    SET preco = 59.90, estoque = 15
    WHERE id_produto = 1
""")

print("Depois do UPDATE:")
spark.sql("""
    SELECT * FROM loja_bronze.produtos WHERE id_produto = 1
""").show(truncate=False)

mouse = spark.sql("""
    SELECT preco, estoque FROM loja_bronze.produtos
    WHERE id_produto = 1
""").first()

from decimal import Decimal
assert mouse.preco == Decimal("59.90")
assert mouse.estoque == 15
assert spark.table("loja_bronze.produtos").count() == 11

Antes do UPDATE:


+----------+------------+-----+-------+------------+
|id_produto|nome_produto|preco|estoque|id_categoria|
+----------+------------+-----+-------+------------+
|1         |Mouse       |49.90|20     |1           |
+----------+------------+-----+-------+------------+



Depois do UPDATE:


+----------+------------+-----+-------+------------+
|id_produto|nome_produto|preco|estoque|id_categoria|
+----------+------------+-----+-------+------------+
|1         |Mouse       |59.90|15     |1           |
+----------+------------+-----+-------+------------+



## 5. Exclusão — DELETE

Repetimos a exclusão da Caneta, identificada por id_produto = 4.
WHERE limita a exclusão a esse registro.

Permanecem dez produtos: o Monitor foi adicionado e a Caneta removida.
A Caneta não aparece nos itens dos pedidos deste conjunto de dados.

In [5]:
print("Antes do DELETE:")
spark.sql("""
    SELECT * FROM loja_bronze.produtos ORDER BY id_produto
""").show(truncate=False)

spark.sql("""
    DELETE FROM loja_bronze.produtos WHERE id_produto = 4
""")

print("Depois do DELETE:")
spark.sql("""
    SELECT * FROM loja_bronze.produtos ORDER BY id_produto
""").show(truncate=False)

assert spark.table("loja_bronze.produtos").count() == 10
assert spark.sql("""
    SELECT * FROM loja_bronze.produtos WHERE id_produto = 4
""").count() == 0
assert spark.sql("""
    SELECT * FROM loja_bronze.produtos WHERE id_produto = 11
""").count() == 1

Antes do DELETE:


+----------+------------+------+-------+------------+
|id_produto|nome_produto|preco |estoque|id_categoria|
+----------+------------+------+-------+------------+
|1         |Mouse       |59.90 |15     |1           |
|2         |Teclado     |120.00|10     |1           |
|3         |Caderno     |19.90 |30     |2           |
|4         |Caneta      |3.50  |100    |2           |
|5         |Headset     |199.90|12     |1           |
|6         |Webcam      |149.90|8      |1           |
|7         |Pendrive    |39.90 |25     |1           |
|8         |Lapiseira   |9.90  |40     |2           |
|9         |Borracha    |2.50  |60     |2           |
|10        |Marcador    |6.90  |35     |2           |
|11        |Monitor     |449.90|7      |1           |
+----------+------------+------+-------+------------+



Depois do DELETE:


+----------+------------+------+-------+------------+
|id_produto|nome_produto|preco |estoque|id_categoria|
+----------+------------+------+-------+------------+
|1         |Mouse       |59.90 |15     |1           |
|2         |Teclado     |120.00|10     |1           |
|3         |Caderno     |19.90 |30     |2           |
|5         |Headset     |199.90|12     |1           |
|6         |Webcam      |149.90|8      |1           |
|7         |Pendrive    |39.90 |25     |1           |
|8         |Lapiseira   |9.90  |40     |2           |
|9         |Borracha    |2.50  |60     |2           |
|10        |Marcador    |6.90  |35     |2           |
|11        |Monitor     |449.90|7      |1           |
+----------+------------+------+-------+------------+



## 6. Histórico das alterações

DESCRIBE HISTORY apresenta as versões e operações registradas
no log de transações do Delta Lake.

In [6]:
spark.sql("DESCRIBE HISTORY loja_bronze.produtos").select(
    "version", "timestamp", "operation"
).show(6, truncate=False)

+-------+-------------------+---------+
|version|timestamp          |operation|
+-------+-------------------+---------+
|4      |2026-10-13 19:35:07|DELETE   |
|3      |2026-10-13 19:35:05|UPDATE   |
|2      |2026-10-13 19:35:04|WRITE    |
|1      |2026-10-13 19:35:02|WRITE    |
|0      |2026-10-13 19:32:10|WRITE    |
+-------+-------------------+---------+



## 7. Resultado final

Conferimos também as tabelas que não foram alteradas e encerramos
a sessão Spark.

In [7]:
esperado = {
    "categorias": 2,
    "clientes": 5,
    "itens_pedido": 10,
    "pedidos": 5,
    "produtos": 10
}

for tabela, quantidade in esperado.items():
    atual = spark.table(f"loja_bronze.{tabela}").count()
    assert atual == quantidade
    print(f"{tabela}: {atual} registros")

print("INSERT validado: Monitor cadastrado.")
print("UPDATE validado: Mouse com preço 59.90 e estoque 15.")
print("DELETE validado: Caneta removida.")
print("As três operações foram realizadas no bucket bronze.")

spark.stop()
print("Sessão Spark encerrada.")

categorias: 2 registros


clientes: 5 registros


itens_pedido: 10 registros


pedidos: 5 registros
produtos: 10 registros
INSERT validado: Monitor cadastrado.
UPDATE validado: Mouse com preço 59.90 e estoque 15.
DELETE validado: Caneta removida.
As três operações foram realizadas no bucket bronze.


Sessão Spark encerrada.
